# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamidrazabajwa49/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Decline-risk classification (forward-looking, not the reactive ranking case).**

I'm choosing classification over opportunity scoring or embeddings/clustering because it's the modeling family I already have the deepest reps in — churn prediction, fraud detection, and spam detection all reduce to the same shape: predict a category from a feature set. Reusing that muscle memory here means less time spent on new tooling and more time on the actual question. Rather than ranking pages that are *already* declining (the opportunity-scoring lane's territory), I want to ask a forward-looking question: using only attributes that exist before a page's trend is known, can a page be flagged as at-risk before it shows up as 'down'? That's a genuinely different decision than prioritizing an already-declining queue — it's prevention, not triage.

In [1]:
# Nothing to compute for this section — it's a reasoning/choice cell.
print("Lane: decline-risk classification")

Lane: decline-risk classification


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Using only content and context attributes that exist independently of a page's 90-day engagement trend (search volume, competition, intent, content type, age, freshness) — not the last-30d/prev-30d performance windows the trend label is built from — can we classify which currently non-declining pages are at elevated risk of moving into 'down', beating a majority-class baseline?

- **Decision it improves:** whether a currently stable/up/new page gets added to a quarterly 'watch list' for a light-touch review, before it becomes a reactive emergency refresh.
- **Who acts on it:** a content strategist running periodic content audits with a fixed review budget — a different actor and a different moment than the editor triaging an already-declining queue.
- **Cost of a wrong call:** two-sided. A false positive — flagging a page that was never actually at risk — burns a limited review slot on something that was fine. A false negative — missing a page about to decline — means it drops into 'down' unnoticed, and by the time it's caught the fix costs more: recovering lost ground instead of just monitoring.
- **Why ML/data helps here:** a hand-rule like 'review anything past 90 days old' is a single signal. If that signal alone doesn't cleanly separate risk (checked below), a rule built on it alone will misfire often — a blended classifier has a real shot at doing better.

In [2]:
# Reasoning cell — no computation needed yet; the case is built with real numbers in Section 3.
print("Decision: whether a non-declining page gets flagged for review before it decays")
print("Actor: content strategist running periodic content audits, fixed review budget")

Decision: whether a non-declining page gets flagged for review before it decays
Actor: content strategist running periodic content audits, fixed review budget


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("Rows, columns:", df.shape)

# Number 1: majority-class baseline — the bar any real classifier has to clear.
vc = df["trend_direction"].value_counts(normalize=True)
majority_label, majority_share = vc.idxmax(), vc.max()
print(f"Majority class '{majority_label}': {majority_share*100:.1f}% — "
      f"a classifier that just guesses '{majority_label}' every time already scores this high.")

# Number 2: does a single freshness signal cleanly separate decline risk?
# (checks whether the naive 'review old content first' rule already does the job)
tier_counts = df["freshness_tier"].value_counts()
down_by_tier = pd.crosstab(df["freshness_tier"], df["trend_direction"], normalize="index")["down"] * 100
print("\nfreshness_tier sizes:\n", tier_counts)
print("\n% of pages currently 'down', by freshness_tier:\n", down_by_tier.round(1))
print(f"\nSpread: {down_by_tier.min():.1f}% to {down_by_tier.max():.1f}% "
      f"(~{down_by_tier.max()-down_by_tier.min():.0f} points) — but two of the four tiers "
      f"hold under 1% of rows each, so most of that spread rests on thin samples.")

# Number 3: data-quality check on a candidate feature (main_intent) before I lean on it.
missing_intent = df["main_intent"].isna().mean() * 100
intent_down = pd.crosstab(df["main_intent"], df["trend_direction"], normalize="index")["down"] * 100
print(f"\nmain_intent missing on {missing_intent:.1f}% of rows")
print("% 'down' by main_intent:\n", intent_down.round(1))

FileNotFoundError: [Errno 2] No such file or directory: '../../data/raw/content_refresh_anonymized.csv'

**What these numbers say:**

- The majority-class baseline is 54.2% — any classifier I build this lane needs to clear that bar by a real margin, or it isn't earning its keep over 'just guess down.'
- Freshness alone spreads from 47.1% to 61.1% down across tiers, but two of the four tiers hold under 1% of rows each, so I'd trust the two well-populated tiers more: 51.1% down at 0–30 days vs. 61.1% at 91–180 days, a real ~10-point gap. Directional, worth keeping as a feature — but not a clean rule on its own.
- `main_intent` is 7.9% missing, and — checked honestly rather than assumed — its decline rate is nearly flat across the three large categories (55.1–57.1% for commercial/informational/transactional). The one standout, navigational at 32.6%, is only 46 rows, so I'm treating that as noise, not signal, until proven otherwise. Intent is going in as a candidate feature, not a headline reason for this lane.
- Net: no single column here separates risk cleanly, which is exactly why a blended classifier is worth the next 7 weeks instead of a hand-written rule.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can say:** any risk label this classifier outputs is *observed, directional, decision-support* — a statement like 'this page sits in a higher-risk group under this model, based on patterns in this 90-day historical slice,' evaluated against a stated baseline (54.2% majority-class) so 'better' has a fixed meaning.

**What I cannot say:** this is not a causal claim — flagging a page doesn't tell me *why* it's at risk, and reviewing it isn't guaranteed to prevent decline. It's also not a forecast of Google's ranking behavior. It's a pattern-matching triage tool trained on one client's 90-day slice, and it needs re-validation — new data, possibly a new client — before I'd trust it outside that window.

In [ ]:
# Reasoning cell — captured in the markdown above.
print("Claims: observed / directional / decision-support only. No causal or Google-behavior claims.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.